# CrewAI Agents with Structured Output

In [ ]:
!pip install -q crewai crewai_tools

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 913.8 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 1.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 1.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 189.8/189.8 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 820.8/820.8 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.9/19.9 MB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 252.5/252.5 kB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.0/48.0 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4

In [ ]:
import crewai
import crewai_tools

print(crewai.__version__)
print(crewai_tools.__version__)

1.15.10
1.15.10


# Set API Keys

In [ ]:
from google.colab import userdata
import os
os.environ["SERPER_API_KEY"] = userdata.get('SERPER_API_KEY')
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY_NEW')
os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')


# Import Dependencies

In [ ]:

from crewai import Agent, Crew, Task, Process, LLM, Process
from crewai_tools import SerperDevTool, DirectoryReadTool
from pydantic import BaseModel
from typing import List

# from dotenv import load_dotenv
# load_dotenv()

#------------------------------------------------------------------------
import warnings
warnings.filterwarnings("ignore")
#------------------------------------------------------------------------


# Define LLMs

In [ ]:
# Create an LLM with a temperature of 0 to ensure deterministic outputs
# -----------
# Create LLM
# -----------

# Create an LLM with a temperature of 0 to ensure deterministic outputs
# OPENAI LLMs
llm = LLM(
         # model="gpt-5.4-mini",
          model="gpt-5.4-nano",
          base_url="https://api.openai.com/v1",
          api_key = os.environ["OPENAI_API_KEY"],
          temperature=0.2)

# # GROQ hosted LLMs
# llm = LLM(
#      model="llama-3.3-70b-versatile",
#      base_url="https://api.groq.com/openai/v1",
#      api_key=os.environ["GROQ_API_KEY"],
#      temperature=0.4)



# Tools

In [ ]:
#------------------------------------------------------------------------
# Create tools
search_tool = SerperDevTool()  # Search capability
docs_tool = DirectoryReadTool(directory='./blog-posts')  # Reads from local files

# Define Output Classes

In [ ]:
#------------------------------------------------------------------------
# Define the Output Class to ensure Structured output from the crew
# This will be used to validate the output of the tasks

class ResearchFindings(BaseModel):
    main_points: List[str]
    key_technologies: List[str]
    societal_impact: str

class Report(BaseModel):
    title: str
    introduction: str
    body: str
    conclusion: str
#------------------------------------------------------------------------


# Define Agents

In [ ]:
# Create Agents
researcher = Agent(
    role='Research Analyst',
    goal='Use available tool to collect information and provide up-to-date technical and social analysis on a given topic',
    backstory='An expert analyst with a keen eye for technical nitty-gritty with a perspective on human vakues.',
    tools=[search_tool],
    llm=llm,
    verbose=False
)

writer = Agent(
    role='Content Writer',
    goal='Craft engaging report about the provided topic',
    backstory='A skilled writer with a passion for technology and its impact on humanity.',
    tools=[docs_tool],
    llm=llm,
    verbose=False
)


# Define Tasks

In [ ]:
#------------------------------------------------------------------------
# Define tasks
research_task = Task(
    description='Research the latest trends in the topic {topic}',
    expected_output=('A summary of recent developments including a unique perspective on their significance.'
        'Your output should contain the following:'
        'main_points: the main textual summary of the report'
        'key_technologies: key technologies enabling the change'
        'societal_impact: how it impacts the life of people and the society as a whole'),
    agent=researcher,
    output_pydantic = ResearchFindings
)

writing_task = Task(
    description=("""Write an engaging report about a topic based on the research analyst's summary.
                    You will receive research output in JSON format from the researcher.
                    You need to extract the following piece of information from the object returned by the `research_task`
                    'main_points: the main textual summary of the report'
                    'key_technologies: key technologies enabling the change'
                    'societal_impact: how it impacts the life of people and the society as a whole'
                     'Use this information to write a comprehensive and engaging report.'
                     """),
    expected_output=(
        "A structured report with title, introduction, body, and conclusion, "
        "written in a clear and engaging style."),
    agent=writer,
    output_pydantic=Report,       # Structured output format
    output_file='blog-posts/report.md',  # The final blog post will be saved here
    # depends_on=[research_task],
    context = [research_task], # Pass research context to writer
    verbose=True
)

# Define Crew (Orchestration Layer)

In [ ]:
#-----------------------------------------------------------------------------
# Assemble a crew with planning enabled
crew = Crew(
    agents=[researcher, writer],
    tasks=[research_task, writing_task],
    verbose=False,
    process=Process.sequential,
    planning=True,  # Enable planning feature
)

# Run the Crew

In [ ]:
# Run the Crew
results = await crew.kickoff_async(
    inputs={"topic": "Social media and its impact on humans"}
    )


╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [33]:
data = results.pydantic
data
# from pprint import pprint
# pprint(data)

Report(title='Social Media in 2026: How Short-Form Video, AI Personalization, and Regulation Are Reshaping Human Attention, Trust, and Community', introduction='Social media’s impact on humans is evolving quickly—and in 2026, the changes are not just about new features, but about how attention is captured, how trust is formed (or eroded), and how communities form. The dominant attention format is shifting toward short-form video, while deeper algorithmic personalization and generative AI accelerate both content creation and discovery. At the same time, social commerce is turning feeds into shopping funnels, and regulation is intensifying—especially around minors, harmful content, and recommender system transparency. These shifts matter now because they reshape how people allocate attention, how quickly information ecosystems can change, and how communities can form often faster than individuals can adapt their habits, coping strategies, and media literacy.', body='Social media in 2026 

AttributeError: 'Report' object has no attribute 'task_output'

In [ ]:
#------------------------------------------------------------
# Verify successful context passing between the agents
#------------------------------------------------------------

# Extract the outputs of individual tasks
research_output = results.tasks_output[0].pydantic
writing_output = results.tasks_output[1].pydantic

# --- Verification Logic ---
# Check if the research output is a valid Pydantic model
if not isinstance(research_output, ResearchFindings):
    print("❌ Research task did not produce a valid ResearchFindings object.")
else:
    print("✅ Research task produced a valid ResearchFindings object.")

    # Get a specific piece of information from the research findings
    # For example, the first main point or a key technology
    key_research_point = research_output.main_points[0] if research_output.main_points else ""
    key_technology = research_output.key_technologies[0] if research_output.key_technologies else ""

    print(f"\nKey research point to check: '{key_research_point}'")
    print(f"\nKey technology to check: '{key_technology}'")

    # Check if the writing output is a valid Pydantic model
    if not isinstance(writing_output, Report):
        print("❌ Writing task did not produce a valid Report object.")
    else:
        print("✅ Writing task produced a valid Report object.")

        # Now, verify if the writer's report contains the information from the researcher
        # This is the core of the verification
        if key_research_point in writing_output.body or key_research_point in writing_output.introduction:
            print("🎉 SUCCESS: The writer's report successfully incorporated the research context!")
        else:
            print("⚠️ The writer's report seems to be missing the key research context.\n\n")



✅ Research task produced a valid ResearchFindings object.

Key research point to check: 'Social media in 2026 is increasingly defined by (1) short-form video as the dominant attention format, (2) deeper algorithmic personalization that optimizes for engagement and “durable attention,” (3) generative AI that accelerates content creation and complicates authenticity, (4) social commerce that turns feeds into shopping funnels (especially via livestream), and (5) intensifying regulation and safety interventions focused on minors, harmful content, and recommender systems transparency. These shifts matter now because they reshape how people allocate attention, how trust is formed (or eroded) in information ecosystems, and how communities form—often faster than individuals can adapt their habits, coping strategies, and media literacy.'

Key technology to check: 'Recommendation and ranking systems (multi-stage recommender pipelines) that optimize for engagement signals, watch time, replays, sh